In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

In [ ]:
test = pd.read_csv('/kaggle/input/llm-classification-finetuning/test.csv')
test.head()

In [ ]:
train = pd.read_csv('/kaggle/input/llm-classification-finetuning/train.csv')
train.head()

In [ ]:
def combine_text(row):
   return f"prompt: {row['prompt']}\nresponse_a: {row['response_a']}\nresponse_b: {row['response_b']}"

In [ ]:
train['text'] = train.apply(combine_text,axis = 1)
test['text'] = test.apply(combine_text,axis = 1)

In [ ]:
def lable(raw):
    if raw['winner_model_a'] == 1:
        return 0
    elif raw['winner_model_b'] == 1:
        return 1
    else :
       return 2
train['label'] = train.apply(lable,axis = 1)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(train['text'], train['label'], test_size=0.2, random_state=42)

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, multi_class='multinomial'))
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
val_probs = pipeline.predict_proba(X_val)
log_loss(y_val, val_probs)

In [ ]:
test_probs = pipeline.predict_proba(test['text'])
test_probs

In [ ]:
submission = pd.DataFrame({
    'id': test['id'],
    'winner_model_a': test_probs[:, 0],
    'winner_model_b': test_probs[:, 1],
    'winner_tie': test_probs[:, 2],
})

In [ ]:
submission.to_csv('submission.csv', index=False)